# Fine-tune Qwen2.5-0.5B for B2B Autocomplete

Upload `train.jsonl` and `val.jsonl` from `autocomplete/finetune/` when prompted.

In [ ]:
!pip install -q --upgrade torchao>=0.16.0
!pip install -q transformers peft datasets accelerate sentencepiece

In [ ]:
from google.colab import files

uploaded = files.upload()
print(f"Uploaded: {list(uploaded.keys())}")

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
OUTPUT_DIR = "./output"
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

dataset = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
print(f"Train: {len(dataset['train']):,}, Val: {len(dataset['validation']):,}")

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=64,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=64,
    learning_rate=3e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()
print("Training complete!")

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained("./output/merged")
tokenizer.save_pretrained("./output/merged")
print("Merged model saved")

In [ ]:
!pip install -q gguf
!git clone --depth=1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp

!python /tmp/llama.cpp/convert_hf_to_gguf.py ./output/merged \
    --outfile ./model-f16.gguf --outtype f16

!cd /tmp/llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j$(nproc)
!/tmp/llama.cpp/build/bin/llama-quantize ./model-f16.gguf ./model-q8.gguf Q8_0

import os
size_mb = os.path.getsize("./model-q8.gguf") / 1024 / 1024
print(f"\nGGUF Q8_0 ready: {size_mb:.0f}MB")

In [ ]:
files.download("./model-q8.gguf")